# NB_bishop_ch09_figures

**Bishop Chapter 9 — Key Figures: Overfitting, Weight Decay, Early Stopping, Double Descent, Residual Connections, Dropout & Learning Curves**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 9 on regularization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_09/NB_bishop_ch09_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 9.1 — Overfitting Demo (Train / Test Loss Curves)

In [ ]:
np.random.seed(0)
epochs = np.arange(1, 201)

# Train loss: monotonically decreasing
train_loss = 1.2 * np.exp(-0.025 * epochs) + 0.05 + 0.008 * np.random.randn(len(epochs))

# Test loss: U-shaped (decreases then rises)
test_loss = (0.9 * np.exp(-0.03 * epochs)
             + 0.15 * (1 - np.exp(-0.015 * epochs))
             + 0.06
             + 0.012 * np.random.randn(len(epochs)))

# Smooth for aesthetics
from numpy.lib.stride_tricks import sliding_window_view
def smooth(y, w=5):
    return np.convolve(y, np.ones(w)/w, mode='same')

train_loss = smooth(train_loss, 7)
test_loss  = smooth(test_loss, 7)

opt_epoch = epochs[np.argmin(test_loss)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, train_loss, color=COLORS['blue'], lw=2, label='Training loss')
ax.plot(epochs, test_loss,  color=COLORS['red'],  lw=2, label='Test loss')

# Shade overfitting region
ax.axvspan(opt_epoch, epochs[-1], alpha=0.07, color=COLORS['red'])
ax.axvline(opt_epoch, ls='--', color=COLORS['green'], lw=1.5,
           label=f'Optimal (epoch {opt_epoch})')

ax.annotate('Overfitting\nregion', xy=(155, 0.22), fontsize=10,
            color=COLORS['red'], ha='center', style='italic')
ax.annotate('Underfitting', xy=(20, 0.85), fontsize=10,
            color=COLORS['blue'], ha='center', style='italic')

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Overfitting: Train vs Test Loss')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig9_1_overfitting_demo')
plt.show()

## Figure 9.2 — Effect of L2 Regularization on Polynomial Fit

In [ ]:
np.random.seed(42)

# Generate sin data (same spirit as Ch1)
N = 15
x_data = np.linspace(0, 1, N)
y_data = np.sin(2 * np.pi * x_data) + 0.25 * np.random.randn(N)
x_dense = np.linspace(0, 1, 300)

M = 11  # high-degree polynomial (easy to overfit with 15 points)

# Design matrix
def design_matrix(x, m):
    return np.vander(x, m + 1, increasing=True)

Phi = design_matrix(x_data, M)
Phi_dense = design_matrix(x_dense, M)

# Fit with different lambda values
lambdas = [0, 1e-6, 1e-3, 1.0]
labels  = ['No reg ($\\lambda=0$)',
            '$\\lambda=10^{-6}$',
            '$\\lambda=10^{-3}$',
            '$\\lambda=1$']
cols = [COLORS['red'], COLORS['amber'], COLORS['green'], COLORS['blue']]

fig, ax = plt.subplots(figsize=(8, 5))

# True function
ax.plot(x_dense, np.sin(2 * np.pi * x_dense), 'k--', lw=1, alpha=0.4, label='True $\\sin(2\\pi x)$')
ax.scatter(x_data, y_data, s=50, zorder=5, color='gray', edgecolor='k', label='Data')

for lam, lab, c in zip(lambdas, labels, cols):
    I = np.eye(M + 1)
    w = np.linalg.solve(Phi.T @ Phi + lam * I, Phi.T @ y_data)
    y_fit = Phi_dense @ w
    ax.plot(x_dense, y_fit, color=c, lw=2, label=lab)

ax.set_ylim(-2.0, 2.0)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title(f'L2 Regularization (Weight Decay) — Degree {M} Polynomial')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig9_2_weight_decay')
plt.show()

## Figure 9.3 — Early Stopping

In [ ]:
np.random.seed(1)
epochs = np.arange(1, 301)

train_loss = 1.5 * np.exp(-0.018 * epochs) + 0.04 + 0.006 * np.random.randn(len(epochs))
val_loss   = (1.2 * np.exp(-0.022 * epochs)
              + 0.20 * (1 - np.exp(-0.012 * epochs))
              + 0.06
              + 0.010 * np.random.randn(len(epochs)))

train_loss = smooth(train_loss, 9)
val_loss   = smooth(val_loss, 9)

best_epoch = epochs[np.argmin(val_loss)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, train_loss, color=COLORS['blue'],  lw=2, label='Training loss')
ax.plot(epochs, val_loss,   color=COLORS['red'],   lw=2, label='Validation loss')
ax.axvline(best_epoch, ls='--', lw=2, color=COLORS['green'],
           label=f'Early stop (epoch {best_epoch})')

ax.annotate('', xy=(best_epoch, 0.12), xytext=(best_epoch + 60, 0.12),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=1.5))
ax.text(best_epoch + 65, 0.12, 'Training continues\nbut val loss rises',
        fontsize=9, va='center', color=COLORS['green'])

# Mark the best point
ax.plot(best_epoch, val_loss[best_epoch - 1], 'o', ms=10, color=COLORS['green'],
        zorder=5, markeredgecolor='k', markeredgewidth=0.8)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early Stopping')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig9_3_early_stopping')
plt.show()

## Figure 9.5 — Double Descent Curve

In [ ]:
# Stylized double descent: test error vs model complexity
complexity = np.linspace(0.1, 5.0, 500)

# Classical U-shape that transitions into modern interpolation regime
threshold = 2.5  # interpolation threshold

# Bias term (decreasing)
bias_term = 1.5 * np.exp(-0.8 * complexity)

# Variance term: peaks sharply at threshold, then decreases
variance_peak = 2.5 * np.exp(-8 * (complexity - threshold)**2)
variance_tail = 0.3 * np.exp(-0.6 * (complexity - threshold)) * (complexity > threshold)

test_error  = bias_term + variance_peak + variance_tail + 0.08
train_error = bias_term * 0.5 * np.exp(-0.5 * complexity) + 0.02
# Train goes to zero at interpolation threshold
train_error[complexity >= threshold] = 0.01

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(complexity, test_error,  color=COLORS['red'],  lw=2.5, label='Test error')
ax.plot(complexity, train_error, color=COLORS['blue'], lw=2, label='Train error', ls='--')

ax.axvline(threshold, ls=':', lw=1.5, color='gray')
ax.text(threshold, ax.get_ylim()[1] * 0.95, 'Interpolation\nthreshold',
        ha='center', va='top', fontsize=9, color='gray',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.8))

# Annotate regimes
ax.text(1.0, 0.6, 'Classical\nregime', ha='center', fontsize=10,
        color=COLORS['blue'], style='italic')
ax.text(4.0, 0.25, 'Modern\ninterpolation\nregime', ha='center', fontsize=10,
        color=COLORS['green'], style='italic')

ax.set_xlabel('Model Complexity (number of parameters / $n$)')
ax.set_ylabel('Error')
ax.set_title('Double Descent')
ax.set_ylim(bottom=0)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig9_5_double_descent')
plt.show()

## Figure 9.6 — Residual Connection Block Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-1.5, 2.5)
ax.set_aspect('equal')
ax.axis('off')

# ── nodes ────────────────────────────────────────────────────
box_kw = dict(boxstyle='round,pad=0.4', lw=1.5)

# Input x
ax.text(0, 0.5, '$\\mathbf{x}$', fontsize=16, ha='center', va='center',
        bbox=dict(**box_kw, fc='#e8edf3', ec=COLORS['blue']))

# F(x) block (two layers inside)
rect = FancyBboxPatch((1.8, -0.2), 2.4, 1.4, boxstyle='round,pad=0.15',
                       fc='#fff3e0', ec=COLORS['amber'], lw=2)
ax.add_patch(rect)
ax.text(3.0, 0.85, 'Weight layer', fontsize=10, ha='center', va='center', color=COLORS['amber'])
ax.text(3.0, 0.45, 'ReLU', fontsize=10, ha='center', va='center', color=COLORS['amber'])
ax.text(3.0, 0.05, 'Weight layer', fontsize=10, ha='center', va='center', color=COLORS['amber'])
ax.text(3.0, 1.55, '$\\mathcal{F}(\\mathbf{x})$', fontsize=13, ha='center',
        va='center', color=COLORS['amber'], fontweight='bold')

# Addition node
circle = plt.Circle((5.5, 0.5), 0.35, fc='#e8f5e9', ec=COLORS['green'], lw=2)
ax.add_patch(circle)
ax.text(5.5, 0.5, '$+$', fontsize=18, ha='center', va='center',
        color=COLORS['green'], fontweight='bold')

# ReLU after addition
ax.text(6.8, 0.5, 'ReLU', fontsize=12, ha='center', va='center',
        bbox=dict(**box_kw, fc='#fce4ec', ec=COLORS['red']))

# Output
ax.text(8.2, 0.5, '$\\mathbf{y}$', fontsize=16, ha='center', va='center',
        bbox=dict(**box_kw, fc='#e8edf3', ec=COLORS['blue']))

# ── arrows (main path) ──────────────────────────────────────
arrow_kw = dict(arrowstyle='->', color=COLORS['blue'], lw=2,
                mutation_scale=15)

ax.annotate('', xy=(1.8, 0.5), xytext=(0.45, 0.5), arrowprops=arrow_kw)
ax.annotate('', xy=(5.15, 0.5), xytext=(4.2, 0.5), arrowprops=arrow_kw)
ax.annotate('', xy=(6.25, 0.5), xytext=(5.85, 0.5), arrowprops=arrow_kw)
ax.annotate('', xy=(7.85, 0.5), xytext=(7.25, 0.5), arrowprops=arrow_kw)

# ── skip connection ─────────────────────────────────────────
# Go up from x, across, then down into +
skip_y = 2.1
ax.annotate('', xy=(0.0, skip_y - 0.1), xytext=(0.0, 0.85),
            arrowprops=dict(arrowstyle='-', color=COLORS['red'], lw=2, ls='--'))
ax.annotate('', xy=(5.5, skip_y - 0.1), xytext=(0.0, skip_y),
            arrowprops=dict(arrowstyle='-', color=COLORS['red'], lw=2, ls='--'))
ax.annotate('', xy=(5.5, 0.85), xytext=(5.5, skip_y),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2, ls='--',
                            mutation_scale=15))

ax.text(2.7, skip_y + 0.2, 'Identity (skip connection)', fontsize=10,
        ha='center', color=COLORS['red'], style='italic')

# Equation
ax.text(4.25, -1.1,
        '$\\mathbf{y} = \\mathrm{ReLU}\\bigl(\\mathcal{F}(\\mathbf{x}) + \\mathbf{x}\\bigr)$',
        fontsize=14, ha='center', va='center')

ax.set_title('Residual Block', fontsize=14, pad=20)

fig.tight_layout()
save_fig(fig, 'fig9_6_residual_connection')
plt.show()

## Figure 9.9 — Dropout Visualization

In [ ]:
np.random.seed(7)

layer_sizes = [4, 6, 6, 3]  # input, hidden1, hidden2, output
layer_labels = ['Input', 'Hidden 1', 'Hidden 2', 'Output']

# Decide which hidden neurons are dropped
dropped = {}
for i in range(1, len(layer_sizes) - 1):  # hidden layers only
    mask = np.random.rand(layer_sizes[i]) < 0.4  # ~40% dropout
    # Ensure at least one is dropped and one survives
    if not mask.any():
        mask[0] = True
    if mask.all():
        mask[-1] = False
    dropped[i] = mask

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for panel, (ax, title, show_dropout) in enumerate(zip(
        axes, ['Standard Network', 'After Dropout (p = 0.4)'], [False, True])):

    ax.set_xlim(-0.5, len(layer_sizes) - 0.5)
    ax.set_ylim(-0.5, max(layer_sizes) + 0.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12)

    positions = {}  # (layer, neuron) -> (x, y)
    for li, n in enumerate(layer_sizes):
        y_offset = (max(layer_sizes) - n) / 2
        for ni in range(n):
            positions[(li, ni)] = (li, y_offset + ni)

    # Draw connections
    for li in range(len(layer_sizes) - 1):
        for ni in range(layer_sizes[li]):
            for nj in range(layer_sizes[li + 1]):
                src_dropped = show_dropout and li in dropped and dropped[li][ni]
                dst_dropped = show_dropout and (li + 1) in dropped and dropped[li + 1][nj]
                if src_dropped or dst_dropped:
                    continue
                x0, y0 = positions[(li, ni)]
                x1, y1 = positions[(li + 1, nj)]
                ax.plot([x0, x1], [y0, y1], color='gray', lw=0.5, alpha=0.4)

    # Draw neurons
    for li, n in enumerate(layer_sizes):
        for ni in range(n):
            x, y = positions[(li, ni)]
            is_dropped = show_dropout and li in dropped and dropped[li][ni]
            if is_dropped:
                # Crossed-out neuron
                ax.plot(x, y, 'o', ms=18, color='lightgray', markeredgecolor='gray',
                        markeredgewidth=1, zorder=5)
                ax.plot([x - 0.15, x + 0.15], [y - 0.15, y + 0.15],
                        color=COLORS['red'], lw=2.5, zorder=6)
                ax.plot([x - 0.15, x + 0.15], [y + 0.15, y - 0.15],
                        color=COLORS['red'], lw=2.5, zorder=6)
            else:
                if li == 0:
                    fc = COLORS['blue']
                elif li == len(layer_sizes) - 1:
                    fc = COLORS['green']
                else:
                    fc = COLORS['amber']
                ax.plot(x, y, 'o', ms=18, color=fc, markeredgecolor='k',
                        markeredgewidth=0.8, zorder=5)

        # Layer label
        ax.text(li, -0.9, layer_labels[li], ha='center', fontsize=9)

fig.tight_layout()
save_fig(fig, 'fig9_9_dropout')
plt.show()

## Figure 9.7 — Learning Curves (High Bias vs High Variance)

In [ ]:
dataset_sizes = np.arange(20, 520, 20)

# ── High-bias model (underfit) ──────────────────────────────
hb_train = 0.35 - 0.12 * (1 - np.exp(-0.008 * dataset_sizes))
hb_test  = 0.45 * np.exp(-0.004 * dataset_sizes) + 0.28

# ── High-variance model (overfit) ──────────────────────────
hv_train = 0.05 + 0.10 * (1 - np.exp(-0.005 * dataset_sizes))
hv_test  = 0.55 * np.exp(-0.006 * dataset_sizes) + 0.12

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: High bias
ax = axes[0]
ax.plot(dataset_sizes, hb_train, color=COLORS['blue'], lw=2, label='Train error')
ax.plot(dataset_sizes, hb_test,  color=COLORS['red'],  lw=2, label='Test error')
ax.axhline(0.22, ls=':', color='gray', lw=1, label='Bayes error')
ax.fill_between(dataset_sizes, hb_train, hb_test, alpha=0.08, color=COLORS['amber'])
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Error')
ax.set_title('High Bias (Underfitting)')
ax.set_ylim(0, 0.6)
ax.annotate('Both converge\nto high error', xy=(400, 0.30), fontsize=9,
            color=COLORS['amber'], ha='center', style='italic')
ax.legend()

# Panel 2: High variance
ax = axes[1]
ax.plot(dataset_sizes, hv_train, color=COLORS['blue'], lw=2, label='Train error')
ax.plot(dataset_sizes, hv_test,  color=COLORS['red'],  lw=2, label='Test error')
ax.axhline(0.08, ls=':', color='gray', lw=1, label='Bayes error')
ax.fill_between(dataset_sizes, hv_train, hv_test, alpha=0.08, color=COLORS['red'])
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Error')
ax.set_title('High Variance (Overfitting)')
ax.set_ylim(0, 0.6)
ax.annotate('Large gap\n(more data helps)', xy=(200, 0.38), fontsize=9,
            color=COLORS['red'], ha='center', style='italic')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig9_learning_curves')
plt.show()

In [ ]:
print('All Chapter 9 figures generated.')